In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('C:/Docs/Healthcare-Payment-Integrity/data/NPPES_Data_Dissemination_November_2026_V2/npidata_pfile_20050523-20261102.csv',
                 nrows=0)

print(df.columns.to_list())

In [ ]:
nppes_cols = [
    # identity
    'NPI', 'Entity Type Code',
    'Provider Last Name (Legal Name)', 'Provider First Name',
    'Provider Middle Name', 'Provider Credential Text',
    # practice location (city/state/zip only)
    'Provider Business Practice Location Address City Name',
    'Provider Business Practice Location Address State Name',
    'Provider Business Practice Location Address Postal Code',
    # dates
    'Provider Enumeration Date', 'Last Update Date',
    'NPI Deactivation Reason Code', 'NPI Deactivation Date',
    'NPI Reactivation Date',
]

# taxonomy: code + primary switch for all 15 slots
for i in range(1, 16):
    nppes_cols.append(f'Healthcare Provider Taxonomy Code_{i}')
    nppes_cols.append(f'Healthcare Provider Primary Taxonomy Switch_{i}')

In [ ]:
print(len(nppes_cols))

In [ ]:
nppes_df = pd.read_csv('C:/Docs/Healthcare-Payment-Integrity/data/NPPES_Data_Dissemination_November_2026_V2/npidata_pfile_20050523-20261102.csv',
                 chunksize=500_000, usecols=nppes_cols, dtype=str, na_filter=False
                 )

nppes_pieces =[]

for chunk in nppes_df:
    in_scope = chunk['Provider Business Practice Location Address State Name'].str.strip().str.upper().isin(['MA','FL','MASSACHUSETTS','FLORIDA'])
    nppes_filtered = chunk[in_scope]

    nppes_pieces.append(nppes_filtered)

nppes_scope = pd.concat(nppes_pieces, ignore_index=True)
print(len(nppes_scope))
print(nppes_scope['Provider Business Practice Location Address State Name'].value_counts())
print(nppes_scope['Entity Type Code'].value_counts())

In [ ]:
test_df = pd.read_csv('C:/Docs/Healthcare-Payment-Integrity/data/NPPES_Data_Dissemination_November_2026_V2/npidata_pfile_20050523-20261102.csv',
                 usecols=nppes_cols, dtype=str, na_filter=False, nrows=500_000
                 )

print(test_df['Provider Business Practice Location Address State Name'].value_counts())

In [ ]:
test1 = test_df['Provider Business Practice Location Address State Name'].str.contains('MASS|FLOR', case=False)
print(test_df[test1]['Provider Business Practice Location Address State Name'].value_counts())

test2 = test_df['Provider Business Practice Location Address State Name'].str.contains('MASSACHUSETTS|FLORIDA', case=False)
print(test_df[test2]['Provider Business Practice Location Address State Name'].value_counts())

In [ ]:
nppes_scope.to_parquet('C:/Docs/Healthcare-Payment-Integrity/data/Processed/nppes_providers.parquet', index=False)

In [ ]:
df = pd.read_parquet('C:/Docs/Healthcare-Payment-Integrity/data/Processed/nppes_providers.parquet')
print(df)

In [ ]:
print(df['NPI'].duplicated().sum())

In [ ]:
print(df['Healthcare Provider Primary Taxonomy Switch_1'].value_counts())

In [ ]:
switch_cols =[]

for col in df.columns:
    if 'Healthcare Provider Primary Taxonomy Switch' in col:
        switch_cols.append(col)

print(len(switch_cols))

is_primary = df[switch_cols] == 'Y'
primary_count = is_primary.sum(axis=1)
print(primary_count.value_counts())